# BANA 4373 / ECON 4370 — Lecture 3 (Data Preparation I: Part B)
## Data Cleaning and Reproducibility in Practice (with the U.S. Census ACS API)

**Goal today:** Build *trustworthy* and *reproducible* data workflows.

We will build on our Lecture 2 API call to the **American Community Survey (ACS) 5-year** and focus on:
- **data types** (strings vs numeric)
- **missingness and suppressed values**
- **sanity checks** (duplicates, ranges, units)
- **reproducibility habits** (clear steps, rerun-ability)

We will use:
- `B19013_001E` = **Median household income (estimate)**
- `B01003_001E` = **Total population (estimate)**

We’ll start with **Texas (state FIPS = 48)** and then extend to **Texas counties**.


## 0. Setup
Run the cell below to import packages.

> If you get `ModuleNotFoundError: requests`, run:  
> `pip install requests` (or `conda install requests`) in your environment.


In [ ]:
import requests
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Cell Finished Running")

## Project structure (created in class)

This project follows a reproducible structure:

- `data_raw/`: original data (never edited)  
- `data_clean/`: cleaned, analysis-ready data  
- `notebooks/`: Jupyter notebooks  
- `output/`: figures, tables, exports  

**Goal:** if we restart the kernel and re-run the notebook, the same folders and outputs should be created again automatically.


In [ ]:
from pathlib import Path

# Base project directory (relative to the notebook location)
project_root = Path("/Users/SHSU/Library/CloudStorage/Dropbox/Classes/ECON_4370/Lecture_3")

folders = [
    "data_raw",
    "data_clean",
    "notebooks",
    "output"
]

for folder in folders:
    (project_root / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created:")
for p in sorted(project_root.iterdir()):
    print(" -", p)

print("Project folders created")

## 1. What does an API return?
An API typically returns **structured data** (often JSON). In many cases, it’s just a **table in disguise**.

### Try in a browser (no code)
Copy/paste this URL into your browser:

```
https://api.census.gov/data/2022/acs/acs5?get=NAME,B19013_001E,B01003_001E&for=state:48
```

You should see:
- First row: column names  
- Next row(s): data values


## 2. Make the same request in Python (Texas only)
Now we will request the same data programmatically.


In [ ]:
base = "https://api.census.gov/data/2022/acs/acs5"
params = {
    "get": "NAME,B19013_001E,B01003_001E",
    "for": "state:48"  # Texas
}

r = requests.get(base, params=params, timeout=30)
r.raise_for_status()

data = r.json()
data[:2]  # header row + first data row


## 3. Convert JSON to a DataFrame
The Census API returns a list of rows:
- `data[0]` = header
- `data[1:]` = records

We’ll convert numeric fields to numbers.


In [ ]:
header = data[0]
rows = data[1:]

df_tx = pd.DataFrame(rows, columns=header)

# Convert numeric columns (they arrive as strings)
for col in ["B19013_001E", "B01003_001E"]:
    df_tx[col] = pd.to_numeric(df_tx[col], errors="coerce")

df_tx


### Interpret the results (quick check)
- Is the income in **dollars**?
- Is population a **count**?
- Do you see any missing values (`NaN`)? Why might that happen?


## 4. Expand to Texas counties (all counties)
Now we request the same variables for **every county in Texas**.

Browser URL (optional to preview):
```
https://api.census.gov/data/2022/acs/acs5?get=NAME,B19013_001E,B01003_001E&for=county:*&in=state:48
```


In [ ]:
params_counties = {
    "get": "NAME,B19013_001E,B01003_001E",
    "for": "county:*",
    "in": "state:48"
}

r = requests.get(base, params=params_counties, timeout=30)
r.raise_for_status()
data_c = r.json()

# Notice: values arrive as STRINGS (object dtype) until we convert them.
df_counties = pd.DataFrame(data_c[1:], columns=data_c[0])

df_counties.head()


## 5. Data types: strings vs numeric (and a quick memory preview)
Even when data comes from a high-quality API, **numbers often arrive as text**.
This can silently break sorting, summaries, merges, and calculations.

We will intentionally do a couple of things *before cleaning* to see the problem clearly.


In [ ]:
# TYPE DEMO (INTENTIONALLY BEFORE CLEANING)
print("Dtypes BEFORE cleaning:")
print(df_counties.dtypes)

# 1) Sorting problem: strings sort alphabetically, not numerically
print("\nTop 10 by income (WRONG if income is a string):")
df_counties.sort_values("B19013_001E", ascending=False).head(10)[["NAME", "B19013_001E"]]



In [ ]:
# 2) Numeric operation problem: this should FAIL (or behave badly) if income is a string
print("\nTry a numeric calculation (this should break before conversion):")
df_counties["B19013_001E"].mean()


In [ ]:
# FIX: Convert strings to numbers (coerce invalid values to NaN)
for col in ["B19013_001E", "B01003_001E"]:
    df_counties[col] = pd.to_numeric(df_counties[col], errors="coerce")

print("Dtypes AFTER cleaning:")
print(df_counties.dtypes)

print("\nTop 10 by income (NOW correct):")
df_counties.sort_values("B19013_001E", ascending=False).head(10)[["NAME", "B19013_001E"]]

print("\nNow mean income works:")
df_counties["B19013_001E"].mean()


What is going? Where is my table?

In a Jupyter notebook:
	•	print() always prints
	•	Only the last expression in a cell is automatically displayed
The mean is being computed but it does not display the talbe because it is no the final expression

Comment out the last two lines and you will see the table. However, there is a better solution. We can explicity display the table. 

In [ ]:
print("\nTop 10 by income (NOW correct):")

top10 = (
    df_counties
    .sort_values("B19013_001E", ascending=False)
    .head(10)[["NAME", "B19013_001E"]]
)

display(top10)

print("\nNow mean income works:")
print(df_counties["B19013_001E"].mean())

print("Cell finished running")

Do you see something weird with the mean? 
Does it have any useful economic meaning? 

In [ ]:
# Check for impossible (negative) income values
df_counties[df_counties["B19013_001E"] < 0][
    ["NAME", "B19013_001E"]
].head(10)

Now we know what there is one county that is creating the problem. We have different options to deal with this. 

1. Treat the negartive values as missing.
2. Do not change the data, rather create a validity flag.

Depending on the your needs both are acceptable. However, it is better to create a flag because it is more transparent.

In [ ]:
# Option 1 treat them as missing 

df_counties.loc[
    df_counties["B19013_001E"] < 0, "B19013_001E"
] = pd.NA

# Now, recompute 

print("Mean income after fixing invalid values:")
print(df_counties["B19013_001E"].mean())

The mean makes sense but the numbers after the decimal point are distracting. Lets round it. 

We also have several options. 

1. Round the number, say to two decimals. This changes the number to only those two decimals.
2. Do not change the name only the formatting of the number and the way it is displayed


In [ ]:
# Change the number

df_counties["B19013_001E"].mean().round(2)

In [ ]:
mean_income = df_counties["B19013_001E"].mean().round(2)
print(mean_income)

In [ ]:
mean_income = df_counties["B19013_001E"].mean().round(2)
print(f"Mean household income (valid counties): ${mean_income:,}")

In [ ]:
# Change the format. 
f"{df_counties['B19013_001E'].mean():,.2f}"

In [ ]:
from IPython.display import display

# Let's now create the flag 
df_counties["income_valid"] = df_counties["B19013_001E"] > 0

# This creates a boolean series: True if positive, False if zero, negative, or missing
display(df_counties[["NAME", "B19013_001E", "income_valid"]].head(10))

display(df_counties["income_valid"].value_counts())

# Replace invalid income values with NA
df_counties.loc[~df_counties["income_valid"], "B19013_001E"] = pd.NA

print("Mean income (valid counties only):")
print(df_counties["B19013_001E"].mean().round(2))
print("Cell finished running")


In [ ]:
# OPTIONAL: quick preview of memory usage (teaser for later in the semester)
df_counties[["NAME", "B19013_001E", "B01003_001E", "state", "county"]].memory_usage(deep=True)


## 6. Basic sanity checks
Before analysis, always check:
- duplicates
- missingness
- ranges / outliers
- units and definitions (metadata)


In [ ]:
# The command below will give the dataset size, number of duplicate rows, 
# and number of missing values per column.

df_counties.shape, df_counties.duplicated().sum(), df_counties.isna().sum()


This dataset has the right size, no duplicate rows, and exactly one missing income value.
That means our structure is correct, but we still need to make a decision about how to handle that one county.

Lets make it a little bit more readable

In [ ]:
print("Shape (rows, columns):", df_counties.shape)
print("Number of duplicate rows:", df_counties.duplicated().sum())
print("\nMissing values by column:")
print(df_counties.isna().sum())

In [ ]:
df_counties[["B19013_001E","B01003_001E"]].describe()


In [ ]:
# Ooops, forgot to round the numbers
df_counties[["B19013_001E","B01003_001E"]].describe().round(2)

## 7. Quick analysis: top counties by median household income
This is just a demo (we will do better visualizations later).


In [ ]:
top10 = df_counties.sort_values("B19013_001E", ascending=False).head(10)
top10[["NAME","B19013_001E","B01003_001E","state","county"]]


## 8. (Optional) Change geography or year
- Change `state:48` to another state FIPS (e.g., 06 = California, 12 = Florida)
- Change the year in the base URL (e.g., 2021, 2023 if available)

### State FIPS examples
- Texas = 48
- California = 06
- Florida = 12
- New York = 36


In [ ]:
# Try another state here by changing the FIPS code.
# Example: California
params_ca = {
    "get": "NAME,B19013_001E,B01003_001E",
    "for": "state:06"
}

data_ca = requests.get(base, params=params_ca, timeout=30).json()
df_ca = pd.DataFrame(data_ca[1:], columns=data_ca[0])
for col in ["B19013_001E", "B01003_001E"]:
    df_ca[col] = pd.to_numeric(df_ca[col], errors="coerce")

df_ca


In [ ]:
# Finally, lets change the name of the columns
# We create a small dictionary 

df_counties = df_counties.rename(columns={
    "B19013_001E": "median_household_income",
    "B01003_001E": "population"
})

df_counties[["median_household_income", "population"]].describe().round(2)


## 9. Short reflection 
1. What surprised you about the API output?
2. What steps did we take to make the data analysis-ready?
3. Why is this process more reproducible than downloading an Excel file?
